# 8장. 작은 데이터 분석 프로젝트 완성하기

이 노트북은 `book/chapters/ch08_midterm_project.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 중간 프로젝트 실습 자료입니다.

이번 장의 핵심은 데이터 불러오기, 전처리, EDA, 시각화, 보고서 작성을 하나의 **재현 가능한 분석 파이프라인**으로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 `data/raw/`의 원본 CSV 4개에서 시작합니다.
- 전처리 결과는 `data/processed/`에 저장합니다.
- 분석 결과표는 `reports/`에 저장합니다.
- 그래프는 `reports/figures/`에 저장합니다.
- 최종 보고서는 `reports/ch08_midterm_report.md`로 저장합니다.
- 보고서에서는 관찰과 원인 가설을 구분합니다.


## 1. 프로젝트 목표

온라인 쇼핑몰 운영자가 최근 주문 데이터를 바탕으로 기본 현황을 파악하려고 한다고 가정합니다. 이번 프로젝트에서는 다음 질문에 답합니다.

| 분석 질문 | 주요 지표 | 결과 형태 |
|---|---|---|
| 카테고리별 매출은 어떻게 다른가? | 총매출, 판매 수량, 매출 비중 | 집계표, 막대그래프 |
| 월별 매출과 주문 수는 어떻게 변하는가? | 월별 매출, 주문 수, 평균 주문 금액 | 집계표, 선그래프 |
| 구매 금액 상위 고객은 누구인가? | 고객별 총 구매 금액, 주문 횟수, 평균 주문 금액 | 집계표, 막대그래프 |
| 주문 상태별 주문 수는 어떻게 분포하는가? | 주문 상태별 주문 수 | 요약표 |

고객 만족도, 광고 효과, 이탈 이유처럼 현재 데이터에 없는 내용은 단정하지 않습니다.


## 2. 프로젝트 분석 흐름

이번 프로젝트는 다음 순서로 진행합니다.

1. 원본 데이터 불러오기
2. 데이터 구조 확인
3. 전처리 수행
4. 분석용 데이터 병합
5. 주요 지표 계산
6. 시각화 생성
7. 결과 파일 저장
8. 보고서 작성
9. LLM을 활용한 검토
10. 최종 결과 정리


## 3. 패키지와 경로 설정

프로젝트 루트에서 실행하는 경우와 `notebooks/` 폴더 안에서 실행하는 경우를 모두 고려해 경로를 설정합니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)
print('그래프 폴더:', FIGURE_DIR)


## 4. 원본 데이터 불러오기

먼저 원본 데이터를 불러옵니다. 파일이 없다면 `python scripts/generate_sample_data.py`를 먼저 실행해야 합니다.


In [ ]:
required_raw_files = [
    RAW_DIR / 'customers.csv',
    RAW_DIR / 'products.csv',
    RAW_DIR / 'orders.csv',
    RAW_DIR / 'order_items.csv',
]

missing_files = [path for path in required_raw_files if not path.exists()]
if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('원본 데이터가 없습니다. 먼저 python scripts/generate_sample_data.py 를 실행하세요.')

customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


## 5. 원본 데이터 구조 확인

데이터 크기, 컬럼명, 결측치, 중복 상태를 모른 채 전처리나 집계를 시작하면 뒤에서 오류를 찾기 어렵습니다. 먼저 구조를 요약합니다.


In [ ]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

dataset_summary = pd.DataFrame([
    {
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_values': int(df.isna().sum().sum()),
        'duplicated_rows': int(df.duplicated().sum()),
    }
    for name, df in raw_data.items()
])

dataset_summary


In [ ]:
for name, df in raw_data.items():
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print('missing values:', df.isna().sum().sum())
    print('duplicated rows:', df.duplicated().sum())


In [ ]:
dataset_summary.to_csv(REPORT_DIR / 'ch08_dataset_summary.csv', index=False, encoding='utf-8-sig')
print('데이터 개요 저장 완료')


## 6. 전처리 함수 준비

원본 DataFrame은 직접 수정하지 않고 복사본을 만들어 전처리합니다. 문자열 공백 제거와 숫자 변환 함수를 먼저 준비합니다.


In [ ]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip(),
        )

    return result


def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce',
    )


## 7. 고객 데이터 전처리

고객 데이터에서는 문자열 공백 제거, 나이 숫자형 변환, 나이 결측치 중앙값 대체, 도시 결측치 `Unknown` 처리, 가입일 날짜형 변환을 수행합니다.


In [ ]:
customers_clean = strip_string_columns(customers)

if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')

customers_clean = customers_clean.drop_duplicates()
customers_clean.head()


## 8. 상품, 주문, 주문 상세 데이터 전처리

상품 가격, 주문 날짜, 주문 상태, 주문 상세 수량/단가를 분석 가능한 형태로 정리합니다. 주문 상세에는 `line_total` 파생 컬럼을 만듭니다.


In [ ]:
products_clean = strip_string_columns(products)

if 'price' in products_clean.columns:
    products_clean['price'] = to_number(products_clean['price'])
    products_clean = products_clean[products_clean['price'] > 0]

products_clean = products_clean.drop_duplicates()

orders_clean = strip_string_columns(orders)

if 'order_date' in orders_clean.columns:
    orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
    orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)

if 'order_status' in orders_clean.columns:
    status_map = {
        'complete': 'completed',
        'Complete': 'completed',
        'COMPLETED': 'completed',
        '완료': 'completed',
        'cancel': 'cancelled',
        'Cancel': 'cancelled',
        'CANCELLED': 'cancelled',
        '취소': 'cancelled',
        'refund': 'refunded',
        'Refund': 'refunded',
        'REFUNDED': 'refunded',
        '환불': 'refunded',
    }
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)

orders_clean = orders_clean.drop_duplicates()

order_items_clean = strip_string_columns(order_items)
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]
order_items_clean['line_total'] = order_items_clean['quantity'] * order_items_clean['unit_price']
order_items_clean = order_items_clean.drop_duplicates()

print('products_clean:', products_clean.shape)
print('orders_clean:', orders_clean.shape)
print('order_items_clean:', order_items_clean.shape)


## 9. 전처리 결과 저장과 전후 비교

전처리 결과를 `data/processed`에 저장하고, 전처리 전후 데이터 크기를 비교합니다.


In [ ]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_summary = pd.DataFrame([
    {'dataset': name, 'rows_processed': df.shape[0], 'columns_processed': df.shape[1]}
    for name, df in processed_data.items()
])

preprocessing_comparison = dataset_summary.merge(processed_summary, on='dataset')
preprocessing_comparison.to_csv(REPORT_DIR / 'ch08_preprocessing_comparison.csv', index=False, encoding='utf-8-sig')
preprocessing_comparison


## 10. 파일 간 키 관계 확인

전처리 과정에서 일부 행을 제외했다면 파일 간 키 관계가 깨지지 않았는지 확인해야 합니다.


In [ ]:
invalid_customers = orders_clean[~orders_clean['customer_id'].isin(customers_clean['customer_id'])]
invalid_orders = order_items_clean[~order_items_clean['order_id'].isin(orders_clean['order_id'])]
invalid_products = order_items_clean[~order_items_clean['product_id'].isin(products_clean['product_id'])]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks.to_csv(REPORT_DIR / 'ch08_relationship_checks.csv', index=False, encoding='utf-8-sig')
relationship_checks


## 11. 분석용 데이터 병합

카테고리별 매출은 주문 상세와 상품 정보를 연결해야 하고, 고객별 구매 금액은 주문 상세, 주문, 고객 정보를 연결해야 합니다. 병합 후에는 행 수와 누락값을 확인합니다.


In [ ]:
sales_items = order_items_clean.merge(products_clean, on='product_id', how='left')
order_sales = order_items_clean.merge(orders_clean, on='order_id', how='left')
customer_sales_base = order_sales.merge(customers_clean, on='customer_id', how='left')

print('order_items_clean:', order_items_clean.shape)
print('sales_items:', sales_items.shape, 'category 누락:', sales_items['category'].isna().sum())
print('order_sales:', order_sales.shape, 'order_date 누락:', order_sales['order_date'].isna().sum())
print('customer_sales_base:', customer_sales_base.shape, 'city 누락:', customer_sales_base['city'].isna().sum())


## 12. 주요 지표 계산

이번 프로젝트의 핵심 분석 질문에 맞춰 카테고리별 매출, 월별 매출, 고객별 구매 금액, 주문 상태별 주문 수를 계산합니다.


In [ ]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales.to_csv(REPORT_DIR / 'ch08_category_sales.csv', index=False, encoding='utf-8-sig')
category_sales


In [ ]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales.to_csv(REPORT_DIR / 'ch08_monthly_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales


In [ ]:
group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.to_csv(REPORT_DIR / 'ch08_customer_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.head(10)


In [ ]:
order_status_summary = orders_clean['order_status'].value_counts(dropna=False).reset_index()
order_status_summary.columns = ['order_status', 'order_count']
order_status_summary['order_ratio'] = (
    order_status_summary['order_count'] / order_status_summary['order_count'].sum() * 100
).round(2)

order_status_summary.to_csv(REPORT_DIR / 'ch08_order_status_summary.csv', index=False, encoding='utf-8-sig')
order_status_summary


## 13. 결과 시각화

핵심 결과를 그래프로 저장합니다. 보고서나 발표 자료에서 재사용하기 쉽도록 PNG 파일로 남깁니다.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(category_sales['category'], category_sales['total_sales'])
plt.title('카테고리별 매출')
plt.xlabel('카테고리')
plt.ylabel('총매출')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ch08_category_sales.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_sales['order_month'], monthly_sales['total_sales'], marker='o')
plt.title('월별 매출 추이')
plt.xlabel('주문 월')
plt.ylabel('총매출')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ch08_monthly_sales.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
top_customers = customer_sales.head(10).copy()
top_customers['customer_label'] = 'Customer ' + top_customers['customer_id'].astype(str)
top_customers = top_customers.sort_values('total_sales')

plt.figure(figsize=(10, 6))
plt.barh(top_customers['customer_label'], top_customers['total_sales'])
plt.title('구매 금액 상위 10명 고객')
plt.xlabel('총 구매 금액')
plt.ylabel('고객')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ch08_top_customers.png', dpi=150, bbox_inches='tight')
plt.show()


## 14. 해석 메모 작성

분석 결과는 숫자와 그래프만으로 끝나지 않습니다. 관찰, 주의사항, 다음 질문을 함께 기록합니다.


In [ ]:
interpretation_notes = pd.DataFrame({
    'analysis': [
        '카테고리별 매출',
        '월별 매출',
        '고객별 구매 금액',
        '주문 상태별 주문 수',
    ],
    'observation': [
        '매출이 높은 카테고리를 확인할 수 있습니다.',
        '시간에 따른 매출 증가와 감소 흐름을 확인할 수 있습니다.',
        '구매 금액이 높은 고객 후보를 확인할 수 있습니다.',
        '주문 상태의 분포를 확인할 수 있습니다.',
    ],
    'caution': [
        '매출이 높은 이유가 판매 수량 때문인지 단가 때문인지 추가 확인이 필요합니다.',
        '매출 변화의 원인을 설명하려면 프로모션, 계절성, 주문 수 변화 확인이 필요합니다.',
        '일회성 고액 구매 고객과 반복 구매 고객을 구분해야 합니다.',
        '취소 주문이 매출 계산에 포함되었는지 확인해야 합니다.',
    ],
})

interpretation_notes.to_csv(REPORT_DIR / 'ch08_interpretation_notes.csv', index=False, encoding='utf-8-sig')
interpretation_notes


## 15. 프로젝트 보고서 저장

마지막으로 분석 목적, 데이터 개요, 전처리, 주요 결과, 해석, 한계점, 다음 단계를 Markdown 보고서로 저장합니다.


In [ ]:
report_text = f'''# Chapter 8 중간 프로젝트 보고서

## 1. 분석 목적

온라인 쇼핑몰 고객, 상품, 주문, 주문 상세 데이터를 사용해 기본 매출 현황과 고객 구매 패턴을 분석했습니다.

## 2. 데이터 개요

```text
{dataset_summary.to_string(index=False)}
```

## 3. 전처리 전후 비교

```text
{preprocessing_comparison.to_string(index=False)}
```

## 4. 주요 분석 질문

1. 카테고리별 매출은 어떻게 다른가?
2. 월별 매출과 주문 수는 어떻게 변하는가?
3. 구매 금액 상위 고객은 누구인가?
4. 주문 상태별 주문 수는 어떻게 분포하는가?

## 5. 카테고리별 매출

```text
{category_sales.to_string(index=False)}
```

## 6. 월별 매출

```text
{monthly_sales.to_string(index=False)}
```

## 7. 구매 금액 상위 고객

```text
{customer_sales.head(10).to_string(index=False)}
```

## 8. 주문 상태별 주문 수

```text
{order_status_summary.to_string(index=False)}
```

## 9. 해석 메모

```text
{interpretation_notes.to_string(index=False)}
```

## 10. 한계점

- 현재 데이터만으로 고객 만족도나 이탈 이유는 분석할 수 없습니다.
- 매출 변화의 원인을 설명하려면 프로모션, 광고, 재고, 계절성 데이터가 추가로 필요합니다.
- 구매 금액 상위 고객은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
- 취소 주문과 환불 주문 처리 기준에 따라 매출 결과가 달라질 수 있습니다.

## 11. 다음 단계

- 카테고리별 판매 수량과 평균 단가를 함께 비교합니다.
- 월별 매출 변동 원인을 추가 데이터와 함께 분석합니다.
- 고객별 구매 금액을 기준으로 고객 세분화를 시도합니다.
- LLM을 활용해 보고서 문장을 보완하되, 데이터에 없는 원인은 단정하지 않습니다.
'''

report_path = REPORT_DIR / 'ch08_midterm_report.md'
report_path.write_text(report_text, encoding='utf-8')

print('중간 프로젝트 보고서 저장 완료:', report_path)


## 16. 소스 모듈로 전체 프로젝트 실행

위에서 단계별로 실행한 프로젝트는 `src/midterm_project.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 다시 실행해 볼 수 있습니다.


In [ ]:
from src.midterm_project import run_midterm_project

project_result = run_midterm_project(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    figure_dir=FIGURE_DIR,
    show_figures=False,
)

project_result['report_path']


## 17. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 8장 프로젝트 전체가 자동으로 실행됩니다.

```bash
python scripts/run_midterm_project.py
```

이 스크립트는 전처리 데이터, 결과 CSV, 그래프, 최종 보고서를 모두 생성합니다.


## 18. LLM 검토 프롬프트 예시

LLM은 프로젝트 검토 파트너로 사용할 수 있습니다. 다만 데이터에 없는 원인을 단정하지 않도록 조건을 명확히 넣어야 합니다.

```text
온라인 쇼핑몰 데이터 분석 프로젝트 결과를 검토해 주세요.

주요 결과:
- 카테고리별 매출표
- 월별 매출표
- 고객별 구매 금액표
- 주문 상태별 주문 수

검토 기준:
1. 현재 데이터로 답할 수 있는 질문인지 확인
2. 관찰과 원인 가설이 구분되어 있는지 확인
3. 데이터에 없는 내용을 단정하지 않았는지 확인
4. 추가로 확인해야 할 분석 질문 제안
5. 보고서 문장을 더 명확하게 다듬기
```


## 19. 최종 산출물 확인

최종적으로 아래 파일들이 생성되었는지 확인합니다.


In [ ]:
expected_outputs = [
    REPORT_DIR / 'ch08_midterm_report.md',
    REPORT_DIR / 'ch08_dataset_summary.csv',
    REPORT_DIR / 'ch08_category_sales.csv',
    REPORT_DIR / 'ch08_monthly_sales.csv',
    REPORT_DIR / 'ch08_customer_sales.csv',
    FIGURE_DIR / 'ch08_category_sales.png',
    FIGURE_DIR / 'ch08_monthly_sales.png',
    FIGURE_DIR / 'ch08_top_customers.png',
]

for path in expected_outputs:
    print(path, 'OK' if path.exists() else 'MISSING')


## 20. 정리

이번 장에서는 다음 내용을 하나의 프로젝트로 연결했습니다.

- 원본 데이터 확인
- 전처리와 원본 보존
- 전처리 전후 비교
- 파일 간 키 관계 확인
- 분석용 데이터 병합
- 카테고리별 매출, 월별 매출, 고객별 구매 금액, 주문 상태별 주문 수 계산
- 핵심 그래프 3개 저장
- 해석 메모 작성
- Markdown 보고서 저장
- `src/midterm_project.py`와 `scripts/run_midterm_project.py`로 재현 가능한 파이프라인 구성

다음 장부터는 머신러닝으로 확장합니다. 하지만 머신러닝도 결국 좋은 데이터와 좋은 질문에서 출발합니다.
